# 🔗 Dos DAGs conectados: uno dispara al otro

Hasta ahora viste DAGs sueltos. Acá vas a ver algo nuevo: **un DAG que, al terminar, dispara a otro DAG.**

Usamos un ejemplo simple de un negocio (ventas), sin relación con los feriados:

```
   ventas_1_generar   ──(dispara)──▶   ventas_2_reporte
   genera las ventas                   lee las ventas y
   del día (un CSV)                    arma un reporte
```

- **`ventas_1_generar`** inventa las ventas del día, las guarda en un archivo, y al final **dispara** al segundo.
- **`ventas_2_reporte`** lee esas ventas y arma un reporte (total + ranking de productos).

> 📖 Notebook para leer y entender. Los archivos reales están en `airflow-docker/dags/ventas_1_generar.py` y `ventas_2_reporte.py`.

## Antes que nada: ¿por qué DOS DAGs y no uno?

Podrías poner todo (generar + reporte) en **un solo DAG** con dos tasks. Y estaría bien.

Pero se usan **DAGs separados** cuando las dos partes son **trabajos distintos que conviene tener independientes**. Por ejemplo:
- Cada uno puede tener **su propio horario** (uno a la mañana, otro a la tarde).
- Cada uno tiene **su propio historial, sus reintentos y sus logs** por separado.
- Si el reporte falla, la generación de ventas ya quedó registrada como exitosa (su trabajo terminó).

> 🧠 Regla mental: si dos cosas son **el mismo trabajo**, son 2 tasks de 1 DAG. Si son **trabajos distintos que se coordinan**, son 2 DAGs que se disparan entre sí.

---
# 📄 DAG 1: `ventas_1_generar` (el que dispara)

Leámoslo entero y después la parte importante.

In [ ]:
with open("./airflow-docker/dags/ventas_1_generar.py") as f:
    print(f.read())

### La parte nueva: `TriggerDagRunOperator`

Todo lo demás ya lo conocés (un `@dag`, una `@task` que genera datos). Lo **nuevo** son estas líneas:
```python
from airflow.operators.trigger_dagrun import TriggerDagRunOperator

disparar_reporte = TriggerDagRunOperator(
    task_id="disparar_reporte",
    trigger_dag_id="ventas_2_reporte",   # <- el dag_id del DAG a disparar
)

generar() >> disparar_reporte            # primero generar, DESPUÉS disparar
```

- **`TriggerDagRunOperator`** es un operador especial: su "trabajo" no es correr código tuyo, sino **disparar otro DAG**.
- **`trigger_dag_id`** dice a **cuál** DAG disparar (por su `dag_id`).
- La dependencia `generar() >> disparar_reporte` asegura el orden: primero se generan las ventas, y **recién después** se dispara el reporte.

> Fijate que `disparar_reporte` es **una task más** dentro del DAG 1. Aparece en la vista Graph como un cuadradito, igual que las otras.

---
# 📄 DAG 2: `ventas_2_reporte` (el que es disparado)

Este es un DAG **normal**. Lo único especial es que **no se corre solo por horario**: espera a que alguien lo dispare.

In [ ]:
with open("./airflow-docker/dags/ventas_2_reporte.py") as f:
    print(f.read())

### El detalle clave: `schedule=None`
```python
@dag(
    dag_id="ventas_2_reporte",
    schedule=None,      # None = NO corre por horario; lo dispara el otro DAG
    ...
)
```
`schedule=None` significa "este DAG no arranca solo nunca". Entonces, ¿cuándo corre? **Cuando el DAG 1 lo dispara.**

El resto es una task normal que lee el CSV y arma el reporte. Nada raro.

> 💡 Ojo: `ventas_2_reporte` **igual lo podés disparar a mano** con el botón ▶️ si querés. Pero si el archivo `ventas.csv` no existe (porque nunca corriste el DAG 1), va a fallar a propósito con un mensaje claro. Probalo: es una buena forma de ver un fallo controlado.

---
## ▶️ Probarlo

En la UI (http://localhost:8080), ambos DAGs ya están activados:

1. Buscá **`ventas_1_generar`** y dale **▶️ (Trigger)**. **Solo a ese.**
2. Entrá a `ventas_1_generar` → vista **Graph**: vas a ver `generar` → `disparar_reporte`.
3. Ahora andá a **`ventas_2_reporte`**: 👀 vas a ver **una corrida nueva que vos NO disparaste**. La disparó el DAG 1.
4. Clic en la task `reporte` (DAG 2) → **Logs**: ahí está el reporte impreso.
5. Se puede ver la dependencia de Dags en la UI: Browse>Dag Dependencies.

> 🎓 **Ese es el momento "ajá":** disparaste UN DAG y se ejecutaron DOS. El segundo arrancó solo porque el primero lo llamó.

### Y para verlo desde acá también
Después de disparar el DAG 1 en la UI, corré esta celda para ver los archivos que produjeron los dos DAGs.

In [ ]:
import os

CSV = "./airflow-docker/data/ventas.csv"           # lo genera el DAG 1
REP = "./airflow-docker/data/reporte_ventas.txt"   # lo genera el DAG 2

if os.path.exists(CSV):
    print("===== ventas.csv (DAG 1) =====")
    print(open(CSV).read())
else:
    print("Todavía no existe ventas.csv. ¿Disparaste ventas_1_generar?")

print()
if os.path.exists(REP):
    print("===== reporte_ventas.txt (DAG 2, disparado solo) =====")
    print(open(REP).read())
else:
    print("Todavía no existe el reporte. Esperá unos segundos y volvé a correr esta celda.")

---
## 🧠 Lo importante para llevarte

1. **Un DAG puede disparar a otro** con `TriggerDagRunOperator` (poniendo el `trigger_dag_id` del que querés disparar).
2. El DAG disparado suele tener **`schedule=None`**: no corre por horario, solo cuando lo llaman.
3. Son **dos corridas separadas**: cada una con su propio historial, reintentos y logs. Están conectadas, pero son independientes.
4. Si el DAG 2 falla, el DAG 1 **igual figura exitoso** — su trabajo (generar + disparar) ya terminó.

### ¿Cuándo NO usar esto?
Si las dos partes son realmente **un solo trabajo** que siempre va junto, es más simple hacer **un DAG con dos tasks** (como tu `mini_elt`). Dividir en dos DAGs tiene sentido cuando querés que sean **independientes** (distinto horario, dueño, o manejo de errores).

> 📌 Existe otra forma más moderna de conectar DAGs, por **eventos** (Datasets/Assets): en vez de "A dispara a B", es "B reacciona cuando A produce un dato". Pero para empezar, `TriggerDagRunOperator` es la más clara e intuitiva.